# Warps, divergence, races and atomics

This notebook goes with chapters 2.3 (warps), 6.1 (divergence), 7.1 (races) and 7.2 (atomics).
You will see a race lose updates, fix it with an atomic, measure the cost of a divergent branch, and sum a warp with
shuffle instructions.

The race and divergence cells need a real GPU to show the effect. The simulator (NUMBA_ENABLE_CUDASIM=1) runs threads
one at a time, so it hides races, and it has no shuffle instructions. The notebook says so where it matters.

In [ ]:
import os
import time
import numpy as np
from numba import cuda

SIM = os.environ.get("NUMBA_ENABLE_CUDASIM") == "1"
BLOCKS, TPB = (4, 32) if SIM else (1024, 256)
print("threads in the grid:", BLOCKS * TPB)

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>import os</code> | The simulator switch. |
| 2 | <code>import time</code> | To time the divergence test. |
| 3 | <code>import numpy as np</code> | NumPy. |
| 4 | <code>from numba import cuda</code> | Numba CUDA. |
| 6 | <code>SIM = os.environ.get(&quot;NUMBA_ENABLE_CUDASIM&quot;) == &quot;1&quot;</code> | True in the simulator. |
| 7 | <code>BLOCKS, TPB = (4, 32) if SIM else (1024, 256)</code> | A large grid on a GPU, a tiny one in the simulator. |
| 8 | <code>print(&quot;threads in the grid:&quot;, BLOCKS * TPB)</code> | How many threads will each try to add 1 to the counter. |

## 1. A race: counter += 1 from many threads

counter[0] += 1 is three steps: read, add, write. Two threads can read the same old value and both write old + 1, so
one update is lost. An atomic add does the three steps as one.

In [ ]:
@cuda.jit
def count_plain(counter):
    counter[0] += 1

@cuda.jit
def count_atomic(counter):
    cuda.atomic.add(counter, 0, 1)

for kernel in (count_plain, count_atomic):
    c = cuda.to_device(np.zeros(1, dtype=np.int32))
    kernel[BLOCKS, TPB](c)
    print(f"{kernel.__name__:13s} counted {c.copy_to_host()[0]:>8d} of {BLOCKS * TPB}")

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>@cuda.jit</code> | Compile as a kernel. |
| 2 | <code>def count_plain(counter):</code> | The racy version. |
| 3 | <code>counter[0] += 1</code> | Read, add one, write back. Not safe when many threads do it at once. |
| 5 | <code>@cuda.jit</code> | Compile as a kernel. |
| 6 | <code>def count_atomic(counter):</code> | The safe version. |
| 7 | <code>cuda.atomic.add(counter, 0, 1)</code> | The hardware does read, add and write as one step that no other thread can split. |
| 9 | <code>for kernel in (count_plain, count_atomic):</code> | Run both kernels the same way. |
| 10 | <code>c = cuda.to_device(np.zeros(1, dtype=np.int32))</code> | A counter that starts at zero. |
| 11 | <code>kernel[BLOCKS, TPB](c)</code> | Every thread in the grid adds one. |
| 12 | <code>print(f&quot;{kernel.__name__:13s} counted {c.copy_to_host()[0]:&gt;8d} of {BLOCKS * TPB}&quot;)</code> | On a GPU the plain version counts far fewer than the number of threads. The simulator runs threads one by one, so both look right there. |

## 2. The cost of divergence

The 32 threads of a warp run one instruction at a time together. If some take the if branch and others the else
branch, the warp runs both branches one after the other with part of its lanes switched off. If the whole warp takes the
same branch, nothing is wasted.

In [ ]:
@cuda.jit
def branchy(out, by_warp):
    i = cuda.grid(1)
    key = (i // 32) if by_warp else i
    v = 0.0
    if key % 2 == 0:
        for k in range(200):
            v += k * 0.5
    else:
        for k in range(200):
            v -= k * 0.25
    out[i] = v

out = cuda.device_array(BLOCKS * TPB, dtype=np.float32)
for by_warp in (False, True):
    branchy[BLOCKS, TPB](out, by_warp)
    cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(20):
        branchy[BLOCKS, TPB](out, by_warp)
    cuda.synchronize()
    ms = (time.perf_counter() - t0) / 20 * 1000
    print("split by warp " if by_warp else "split by thread", f"{ms:.3f} ms")

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>@cuda.jit</code> | Compile as a kernel. |
| 2 | <code>def branchy(out, by_warp):</code> | out receives a result per thread. by_warp chooses how threads pick a branch. |
| 3 | <code>i = cuda.grid(1)</code> | Global index. |
| 4 | <code>key = (i // 32) if by_warp else i</code> | If by_warp, all 32 threads of a warp share the same key. Otherwise neighbours get different keys. |
| 5 | <code>v = 0.0</code> | A value to work on. |
| 6 | <code>if key % 2 == 0:</code> | Even keys take the first branch. |
| 7 | <code>for k in range(200):</code> | Some real work, 200 steps. |
| 8 | <code>v += k * 0.5</code> | Add. |
| 9 | <code>else:</code> | Odd keys take the second branch. |
| 10 | <code>for k in range(200):</code> | The same amount of work. |
| 11 | <code>v -= k * 0.25</code> | Subtract. |
| 12 | <code>out[i] = v</code> | Store, so the compiler cannot skip the work. |
| 14 | <code>out = cuda.device_array(BLOCKS * TPB, dtype=np.float32)</code> | Output array, one value per thread. |
| 15 | <code>for by_warp in (False, True):</code> | Try both ways of splitting. |
| 16 | <code>branchy[BLOCKS, TPB](out, by_warp)</code> | Warm up launch, which compiles the kernel for this argument type. |
| 17 | <code>cuda.synchronize()</code> | Wait for it. |
| 18 | <code>t0 = time.perf_counter()</code> | Start the clock. |
| 19 | <code>for _ in range(20):</code> | Twenty launches. |
| 20 | <code>branchy[BLOCKS, TPB](out, by_warp)</code> | Same launch each time. |
| 21 | <code>cuda.synchronize()</code> | Wait for all of them. |
| 22 | <code>ms = (time.perf_counter() - t0) / 20 * 1000</code> | Average time per launch. |
| 23 | <code>print(&quot;split by warp &quot; if by_warp else &quot;split by thread&quot;, f&quot;{ms:.3f} ms&quot;)</code> | Split by thread should take about twice as long: every warp runs both branches. |

## 3. Warp shuffle: sum 32 values without shared memory

shfl_down_sync lets a thread read a register of another thread in the same warp. After five steps (offsets 16, 8, 4,
2, 1) lane 0 holds the sum of the warp. No shared memory and no __syncthreads are needed.
This cell needs a real GPU, because the Numba simulator has no shuffle instructions.

In [ ]:
@cuda.jit
def warp_sum(x, out):
    i = cuda.grid(1)
    v = x[i]
    offset = 16
    while offset > 0:
        v += cuda.shfl_down_sync(0xFFFFFFFF, v, offset)
        offset //= 2
    if cuda.laneid == 0:
        out[i // 32] = v

if not SIM:
    xs = np.arange(BLOCKS * TPB, dtype=np.float32)
    sums = cuda.device_array(BLOCKS * TPB // 32, dtype=np.float32)
    warp_sum[BLOCKS, TPB](cuda.to_device(xs), sums)
    print("warp sums match:", np.allclose(sums.copy_to_host(), xs.reshape(-1, 32).sum(axis=1)))
else:
    print("skipped: the simulator has no shuffle instructions")

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>@cuda.jit</code> | Compile as a kernel. |
| 2 | <code>def warp_sum(x, out):</code> | x in, out gets one sum per warp. |
| 3 | <code>i = cuda.grid(1)</code> | Global index. |
| 4 | <code>v = x[i]</code> | Each lane holds one value in a register. |
| 5 | <code>offset = 16</code> | First step: add the value from 16 lanes further on. |
| 6 | <code>while offset &gt; 0:</code> | Five steps: 16, 8, 4, 2, 1. |
| 7 | <code>v += cuda.shfl_down_sync(0xFFFFFFFF, v, offset)</code> | Read v from lane (my lane + offset) and add it. The mask 0xFFFFFFFF says all 32 lanes take part. |
| 8 | <code>offset //= 2</code> | Halve the offset. |
| 9 | <code>if cuda.laneid == 0:</code> | laneid is this thread's position in its warp, 0 to 31. Lane 0 now holds the full sum. |
| 10 | <code>out[i // 32] = v</code> | Write one result per warp. |
| 12 | <code>if not SIM:</code> | Only on a real GPU. |
| 13 | <code>xs = np.arange(BLOCKS * TPB, dtype=np.float32)</code> | Numbers 0, 1, 2 and so on, so the sums are easy to check. |
| 14 | <code>sums = cuda.device_array(BLOCKS * TPB // 32, dtype=np.float32)</code> | One output per warp. |
| 15 | <code>warp_sum[BLOCKS, TPB](cuda.to_device(xs), sums)</code> | Launch. |
| 16 | <code>print(&quot;warp sums match:&quot;, np.allclose(sums.copy_to_host(), xs.reshape(-1, 32).sum(axis=1)))</code> | Compare with NumPy, which sums each group of 32. |
| 17 | <code>else:</code> | In the simulator. |
| 18 | <code>print(&quot;skipped: the simulator has no shuffle instructions&quot;)</code> | Say why the cell did nothing. |

## 4. Try these

1. In branchy, change the key to (i // 16). Predict the time before you run it.
2. Combine the warp sum with one atomic per warp to sum a whole array. Count the atomics for N = 2 to the power 24.
3. Make the race worse or better by changing the number of blocks. Explain the counts you see.